In [8]:
import pandas as pd
import json
import numpy as np
import polars as pl
import requests
import plotly.express as px
import time
import datetime
import sys
import os
import matplotlib
import http.client
import urllib.parse
import pickle
import httpx
import csv

from tqdm.notebook import tqdm_notebook as tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pprint import pprint
from io import StringIO
from functools import partial

from requests.adapters import ConnectionError, ReadTimeout, ReadTimeoutError
from urllib3.connection import NewConnectionError
from urllib3.util.retry import MaxRetryError


# Settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 100
pd.options.display.width = 20000
pd.set_option('display.float_format', lambda x: '%.4f' % x)
np.set_printoptions(suppress=True)
pl.Config.set_tbl_rows(100)

from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [9]:
def make_request(time_data: str, asset: str, request: str, params: dict):
    """

    :param time_data: at_time | hist | bulk_hist | list/...
    :param asset: stock | option | index
    :return: dict or raw csv
    """
    
    headers = {'Accept': "application/json"}
    url = f"http://127.0.0.1:25510/v2/{time_data}/{asset}/{request}".rstrip('/')

    success = False
    while not success:
        try:
            data = requests.get(url, headers=headers, params=params, timeout=60)
            if data.status_code == 474:
                if 'root' in params.keys():
                    print(params['root'])
                print("474 Connection lost to Theta Data. It's going to sleep to 5 sec")
                time.sleep(5)
            elif ('Next-page' not in data.headers) and (data.status_code != 472): 
                if 'root' in params.keys():
                    print(params['root'])
                print("Next-page' not in the data.headers or error in status code. It's going to sleep to 5 sec")
                print(data)
                print(data.url)
                print(data.status_code)
                print(data.content)
                time.sleep(5)
            else:
                success = True
        except (ConnectionError, NewConnectionError, MaxRetryError, ReadTimeout, ReadTimeoutError) as e:
            print(f"--------{params['root']}--------")
            print(e)
            time.sleep(5)
        except Exception as e:
            print(e)
            raise Exception
    try:
        if data.status_code == 471: 
            raise Exception(data.text)
        elif data.status_code == 472:
            return None, data.status_code
        elif data.status_code == 475:
            print(f"The server incorrect compile: {data.url}")
            return None, data.status_code
    
        if data.status_code != 200:
            print(params['root'], data.status_code, data.text)
    
        if ('use_csv' in list(params.keys())) and (params['use_csv'] == 'True'):
            if data.headers['Next-page'] != 'null':
                raise Exception(f"{params['root']} has an additional pages.")
                
            return data.content.decode('utf-8'), data.status_code
            
        else:
            return data.json(), data.status_code
    except Exception as e:
        print(e)
        print(url)
        print(params)
        raise Exception

# 04.2024 Theta Data 

## Funcs

In [ ]:
# Get stock  EOD
# params = {
#     'root': 'AAPL', 
#     'start_date': '20240102', 
#     'end_date': '20240202'
# }
# make_request('hist', 'option', 'eod', params)

# Get all symbols
# params = {}
# ans = make_request('list/roots', 'index', '', params)

# Get all expiration
# params = {'root': 'AAPL'}
# ans = make_request('list/expirations', '', '', params)

# Get all strikes
params = {'root': 'AAPL', 'exp': '20171103'}
ans = make_request('list/strikes', '', '', params)

# Get all trade days
params = {'root': 'AAPL'}
ans = make_request('list/dates', 'stock', 'quote', params)

# Get all trade days
params = {'root': 'AAPL,SPY,AMD', 'start_date': '20230512'}
ans = make_request('list/contracts', 'option', 'trade', params)

# Get all contracts
params = {'root': 'AAPL,SPY', 'start_date': '20230512'}
ans = make_request('list/contracts', 'option', 'quote', params)

# pd.DataFrame(data=ans['response'], columns=ans['header']['format'])

## Get all tickers' expirations.

In [10]:
ans = make_request('list/roots', 'stock', '', {})
tickers_list = ans[0]['response']

In [11]:
def fetch_expiration(t):
    params = {'root': t}
    ans, status = make_request('list/expirations', '', '', params)
    if ans == None:
        return t, []
    else:
        return t, ans['response']
    

tickers_all_exp = {}
with ThreadPoolExecutor(max_workers=24) as executor:
    futures = {executor.submit(fetch_expiration, t): t for t in tickers_list}

    for future in tqdm(as_completed(futures), total=len(tickers_list)):
        t, response = future.result()
        tickers_all_exp[t] = response

tickers_all_exp = {t: {'expirations': values} for t, values in tickers_all_exp.items()}
tickers_all_exp = {t: values for t, values in tickers_all_exp.items() if len(values['expirations']) > 10}

""" Ошибки с ['ZWZZT', 'ZXIET', 'ZWRKW', 'ZYBT', 'ZY', 'ZZZ', 'ZYNE', 'ZXZZT', 'ZWS', 'ZYME', 'ZXYZ.A', 'ZYXI', 'ZZK']. Чтобы работало, надо:

elif ('Next-page' not in data.headers) and (data.status_code != 472): 
    if 'root' in params.keys():
        print(params['root'])
    print("Next-page' not in the data.headers or error in status code. It's going to sleep to 5 sec")
    print(data)
    print(data.url)
    print(data.status_code)
    print(data.content)
    time.sleep(5)

    return None, data.status_code
"""

  0%|          | 0/22696 [00:00<?, ?it/s]

' Ошибки с [\'ZWZZT\', \'ZXIET\', \'ZWRKW\', \'ZYBT\', \'ZY\', \'ZZZ\', \'ZYNE\', \'ZXZZT\', \'ZWS\', \'ZYME\', \'ZXYZ.A\', \'ZYXI\', \'ZZK\']. Чтобы работало, надо:\n\nelif (\'Next-page\' not in data.headers) and (data.status_code != 472): \n    if \'root\' in params.keys():\n        print(params[\'root\'])\n    print("Next-page\' not in the data.headers or error in status code. It\'s going to sleep to 5 sec")\n    print(data)\n    print(data.url)\n    print(data.status_code)\n    print(data.content)\n    time.sleep(5)\n\n    return None, data.status_code\n'

## Get all tickers' trade days

In [12]:
def trade_dates(t: str):
    params = {'root': t}
    ans, status = make_request('list/dates', 'stock', 'quote', params)
    if status == 472:
        return t, []
        
    trade_dates = pd.Series(ans['response']).astype(str).to_list()
    return t, trade_dates


with ThreadPoolExecutor(max_workers=24) as executor:
    futures = {executor.submit(trade_dates, t): t for t in tickers_all_exp.keys()}

    for future in tqdm(as_completed(futures), total=len(tickers_all_exp.keys())):
        t, response = future.result()
        tickers_all_exp[t]['trade_dates'] = response

tickers_all_exp = {t: values for t, values in tickers_all_exp.items() if len(values['trade_dates']) > 200}
with open('data/all_tickers_expir.pickle', 'wb') as f:
    pickle.dump(tickers_all_exp, f)

  0%|          | 0/8648 [00:00<?, ?it/s]

## Get tickers prices

In [ ]:
def get_stock_prices(t: str, trade_dates: list):
    months_list = pd.DataFrame(pd.to_datetime(trade_dates))
    months_list = months_list[months_list[0] >= '20160101']
    dates_start = months_list.groupby(months_list[0].dt.strftime('%Y%m')).min()[0].values
    dates_end = months_list.groupby(months_list[0].dt.strftime('%Y%m')).max()[0].values

    final_df = pl.DataFrame()
    for start, end in tqdm(zip(dates_start, dates_end)):   
        start = pd.Timestamp(start).strftime('%Y%m%d')
        end = pd.Timestamp(end).strftime('%Y%m%d')

        params = {'root': t, 'use_csv': 'True', 'start_date': start, 'end_date': end}
        ans, status = make_request('hist', 'stock', 'eod', params)
        if status == 472:
            print(f"{t} don't have data {start_date} - {end_date}")
            continue
        elif status == 475:
            continue
    
        cur_df = pl.read_csv(StringIO(ans))\
            .sort('date')\
            .with_columns(pl.lit(t).alias('ticker'))
        final_df = pl.concat([final_df, cur_df])

    return final_df


with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

for core, folder, files in os.walk('data\stocks\daily'):
    already_files = set([f.split('_')[2] for f in files])
    
tickers_all_exp = {t: value for t, value in tickers_all_exp.items() if t not in already_files}
with ThreadPoolExecutor(max_workers=24) as executor:
    futures = {
        executor.submit(get_stock_prices, t, values['trade_dates']): 
        (t, values['trade_dates']) 
        for t, values in tickers_all_exp.items()
    }
    for future in tqdm(as_completed(futures), total=len(tickers_all_exp.keys())):
        cur_df = future.result()
        if (cur_df is None) or len(cur_df) == 0: 
            continue

        try:
            cur_df = cur_df.drop([
                    'bid_size', 'bid_exchange', 'bid', 'bid_condition',
                    'ask_size', 'ask_exchange', 'ask', 'ask_condition'
                ]).with_columns(pl.col('date').cast(pl.String).str.strptime(pl.Date, '%Y%m%d', strict=False))\
                .sort('date')\
                .with_columns(rol_50_vol=pl.col('volume').rolling_mean(window_size=50, min_periods=1))
            ticker = cur_df['ticker'].unique()[0]
            
            min_date = cur_df['date'].min().strftime('%Y%m%d')
            max_date = cur_df['date'].max().strftime('%Y%m%d')
            cur_df.write_parquet(f'data/stocks/daily/{min_date}_{max_date}_{ticker}_daily.parquet')
        except Exception as e:
            print(cur_df)
            print(e)
            raise Exception

## Get EOD options data

In [3]:
stocks_df = pl.read_parquet('data/20230601_stocks_data.parquet')\
    .with_columns(pl.col('date').cast(pl.String).str.to_date('%Y%m%d'))

need_tickers = stocks_df\
    .filter(pl.col('date') < datetime.date(2023, 7, 1))\
    .group_by('ticker').agg(pl.col('volume').mean(), pl.col('count').count())\
    .filter(pl.col('volume') > 500_000)\
    ['ticker'].to_list()

In [4]:
with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

drop_tickers = [t for t in tickers_all_exp.keys() if t in set(need_tickers) and tickers_all_exp[t]['trade_dates'] == []]
for t in drop_tickers:
    del tickers_all_exp[t]

tickers_all_exp_filt = {t: tickers_all_exp[t] for t in tickers_all_exp.keys() if t in set(need_tickers)}

In [5]:
already_tickers = []
for root, direc, files in os.walk('data/options/'):
    for f in files:
        if f.endswith('.parquet') and '20210101' in f:
            f = f.split('_')[2]
            already_tickers.append(f)

for t in already_tickers:
    del tickers_all_exp_filt[t]

need_tickers = tickers_all_exp_filt.keys()

In [6]:
start_range = '20210101'
end_range = '20211231'

dates_start = pd.date_range(start_range, end_range, freq='MS').strftime('%Y%m%d')[::-1]
dates_end = pd.date_range(start_range, end_range, freq='ME').strftime('%Y%m%d')[::-1]
for t in tqdm(need_tickers, total=len(need_tickers)):
    cur_df = pl.DataFrame()
    
    start_time = time.time()
    for start_date, end_date in tqdm(zip(dates_start, dates_end)):
        params = {
            'root': t, 'exp': 0,
            'start_date': start_date, 'end_date': end_date,  
            'use_csv': 'True'
        }
        ans, status_code = make_request('bulk_hist', 'option', 'eod', params)
        if status_code == 472:
            print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
            continue
        elif status_code == 572:
            print(f"\033[1;91m {t} an error \033[0m")
            continue

        ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))
        cur_df = pl.concat([cur_df, ans_df])

    print(t, round(time.time() - start_time, 2))
    if len(cur_df) == 0:
        continue

    cur_df = cur_df.sort('date', 'expiration', 'right', 'strike')
    cur_df.write_parquet(f'data/options/{start_range}_{end_range}_{t}_opts.parquet')

  0%|          | 0/1911 [00:00<?, ?it/s]

0it [00:00, ?it/s]

 AAC 20211201 - 20211231 don't have data 
 AAC 20211101 - 20211130 don't have data 
 AAC 20211001 - 20211031 don't have data 
 AAC 20210901 - 20210930 don't have data 
 AAC 20210801 - 20210831 don't have data 
 AAC 20210701 - 20210731 don't have data 
 AAC 20210601 - 20210630 don't have data 
 AAC 20210501 - 20210531 don't have data 
 AAC 20210401 - 20210430 don't have data 
 AAC 20210301 - 20210331 don't have data 
 AAC 20210201 - 20210228 don't have data 
 AAC 20210101 - 20210131 don't have data 
AAC 215.12


0it [00:00, ?it/s]

 ABCM 20211201 - 20211231 don't have data 
 ABCM 20211101 - 20211130 don't have data 
 ABCM 20211001 - 20211031 don't have data 
 ABCM 20210901 - 20210930 don't have data 
 ABCM 20210801 - 20210831 don't have data 
 ABCM 20210701 - 20210731 don't have data 
 ABCM 20210601 - 20210630 don't have data 
 ABCM 20210501 - 20210531 don't have data 
 ABCM 20210401 - 20210430 don't have data 
 ABCM 20210301 - 20210331 don't have data 
 ABCM 20210201 - 20210228 don't have data 
 ABCM 20210101 - 20210131 don't have data 
ABCM 215.66


...

0it [00:00, ?it/s]

 EXFY 20211201 - 20211231 don't have data 
 EXFY 20211101 - 20211130 don't have data 
 EXFY 20211001 - 20211031 don't have data 
 EXFY 20210901 - 20210930 don't have data 
 EXFY 20210801 - 20210831 don't have data 
 EXFY 20210701 - 20210731 don't have data 
 EXFY 20210601 - 20210630 don't have data 
 EXFY 20210501 - 20210531 don't have data 
 EXFY 20210401 - 20210430 don't have data 
 EXFY 20210301 - 20210331 don't have data 
 EXFY 20210201 - 20210228 don't have data 
 EXFY 20210101 - 20210131 don't have data 
EXFY 253.51


0it [00:00, ?it/s]

 FBIN 20211201 - 20211231 don't have data 
 FBIN 20211101 - 20211130 don't have data 
 FBIN 20211001 - 20211031 don't have data 
 FBIN 20210901 - 20210930 don't have data 
 FBIN 20210801 - 20210831 don't have data 
 FBIN 20210701 - 20210731 don't have data 


KeyboardInterrupt: 

In [23]:
cur_df

shape: (0, 0)
┌┐
╞╡
└┘

In [7]:
minute_1 = 60000
minute_1 * 60 * 16, 62917076 / minute_1 / 60

(57600000, 17.47696555555556)

In [6]:
62638161 / minute_1 / 60 // 15, (57000000 / minute_1 / 60 % 15) * 60

(1.0, 50.000000000000036)

In [211]:
minute_1 = 60000
minute_1 * 60 * 16

57600000

## Create united options file

In [4]:
year: str = '2022'
chunk_size = 500
chunk_counter = 0
batch_dfs = []

for root, dirs, files in os.walk('data/options/'):
    parquet_files = [f for f in files if f.endswith('.parquet') and year in f]
    num_files = len(parquet_files)

    for i in tqdm(range(0, num_files, chunk_size)):
        total_df = pl.DataFrame()
        
        for file in tqdm(parquet_files[i:i + chunk_size]):
            try:
                cur_df = pl.read_parquet(os.path.join(root, file), low_memory=True)\
                    .drop(['root', 'ms_of_day2', 'open', 'high', 'low'])
                total_df = pl.concat([total_df, cur_df])
            except Exception as e:
                print(file)
                print(e)

        # Save intermediate result
        total_df = total_df.with_columns(
            [pl.col(col).cast(pl.String).str.to_date('%Y%m%d') for col in ['expiration', 'date']],
        ).with_columns(
            [pl.col(col).cast(pl.Int16) for col in ['bid_exchange', 'bid_condition', 'ask_exchange', 'ask_condition']],
        ).with_columns(
            [pl.col(col).cast(pl.Int32) for col in ['ms_of_day', 'volume', 'count', 'bid_size', 'ask_size']],
        ).with_columns(
            [pl.col(col).cast(pl.Float32) for col in ['close', 'bid', 'ask']],
        )
        
        batch_file = f'data/temp//batch_{year}_{chunk_counter}.parquet'
        total_df.write_parquet(batch_file)
        batch_dfs.append(batch_file)
        chunk_counter += 1

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/169 [00:00<?, ?it/s]

In [3]:
qqq_df = pl.read_parquet('data/options/20230101_QQQ_opts.parquet')\
    .drop(['root', 'ms_of_day2', 'open', 'high', 'low'])
iwm_df = pl.read_parquet('data/options/20230101_IWM_opts.parquet')\
    .drop(['root', 'ms_of_day2', 'open', 'high', 'low'])
total_df = pl.concat([qqq_df, iwm_df])

batch_file = f'data/temp//batch_6.parquet'
total_df.write_parquet(batch_file)

In [57]:
ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))

## Load Stocks 1 minute

In [7]:
def get_stock_1m(t: str, start_date: str, end_date: str):
    if end_date < '20160101':
        return None

    params = {
        'root': t, 'exp': 0,
        'start_date': start_date, 'end_date': end_date,  
        'use_csv': 'True', 'ivl': '60000'
    }
    ans, status_code = make_request('hist', 'stock', 'ohlc', params)
    
    if status_code == 472:
        print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
        return None
    elif status_code == 572:
        print(f"\033[1;91m {t} an error \033[0m")
        return None
    
    ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))

    return ans_df

In [8]:
liquid_tickers = []
for core, folder, files in os.walk('data\stocks\daily'):
    for file in tqdm(files):
        cur_df = pl.read_parquet(os.path.join(core, file))
        if (cur_df['rol_50_vol'] > 1_000_000).any():
            liquid_tickers.append(file.split('_')[2])

for core, folder, files in os.walk('data/stocks/m1'):
    already_tickers = [file.split('_')[2] for file in files]

tickers_for_load = set(liquid_tickers) - set(already_tickers)

  0%|          | 0/7804 [00:00<?, ?it/s]

In [5]:
with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

tickers_dates = {}
for t in tqdm(tickers_for_load):
    tickers_dates[t] = {}
    
    cur_dates = tickers_all_exp[t]['trade_dates']
    cur_dates = pd.Series(cur_dates, index=cur_dates)[f'20160101':].values
    start_range = cur_dates[0]
    end_range = cur_dates[-1]

    weekly_start = list(pd.date_range(start_range, end_range, freq='W-MON').strftime('%Y%m%d')[::-1])
    weekly_end = list(pd.date_range(start_range, end_range, freq='W-SUN').strftime('%Y%m%d')[::-1])
    weekly_end.extend([end_range])
    weekly_end = np.sort(weekly_end)[::-1]

    tickers_dates[t]['weekly_start'] = weekly_start
    tickers_dates[t]['weekly_end'] = weekly_end

0it [00:00, ?it/s]

In [6]:
for t in tqdm(tickers_for_load, total=len(tickers_for_load)):
    cur_data = tickers_dates[t]
    
    final_df = pl.DataFrame()    
    with ThreadPoolExecutor(max_workers=24) as executor:
        futures = {
            executor.submit(get_stock_1m, t, w_s, w_e): 
            (t, w_s, w_e) for w_s, w_e in zip(cur_data['weekly_start'], cur_data['weekly_end'])
        }
        for future in tqdm(as_completed(futures), total=len(cur_data['weekly_start'])):
            try:
                cur_df = future.result()
                if (cur_df is None) or len(cur_df) == 0: 
                    continue
                final_df = pl.concat([final_df, cur_df])
            except Exception as e:
                print(e)
                raise Exception

    try:
        if (final_df is None) or len(final_df) == 0: 
            continue
    
        final_df = final_df.sort('date')
        min_date = str(final_df['date'].min())
        max_date = str(final_df['date'].max())
        final_df.write_parquet(f'data/stocks/m1/{min_date}_{max_date}_{t}_opts.parquet')
    except Exception as e:
        print(e)
        raise Exception

0it [00:00, ?it/s]

# Options EOD

In [7]:
def get_option_eod(t: str, start_date: str, end_date: str):
    if end_date < '20160101':
        return None
        
    params = {
        'root': t, 'exp': 0,
        'start_date': start_date, 'end_date': end_date,  
        'use_csv': 'True'
    }
    ans, status_code = make_request('bulk_hist', 'option', 'eod', params)
    
    if status_code == 472:
        print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
        return None
    elif status_code == 572:
        print(f"\033[1;91m {t} an error \033[0m")
        return None
    
    ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))
    
    return ans_df

In [8]:
load_year: str = '2020'

liquid_tickers = []
for core, folder, files in os.walk('data\stocks\daily'):
    for file in tqdm(files):
        cur_df = pl.read_parquet(os.path.join(core, file))
        if (cur_df['rol_50_vol'] > 1_000_000).any():
            liquid_tickers.append(file.split('_')[2])

already_tickers = [file.split('_')[2] for file in os.listdir('data/options') if load_year in file]
tickers_for_load = set(liquid_tickers) - set(already_tickers)
len(tickers_for_load)

  0%|          | 0/7804 [00:00<?, ?it/s]

1681

In [9]:
with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)
    
for t in tqdm(tickers_for_load, total=len(tickers_for_load)):
    dates = tickers_all_exp[t]['trade_dates']
    dates = pd.Series(dates, index=dates)[f'{load_year}0101':f'{load_year}1231'].values
    if len(dates) == 0:
        continue

    expirations = tickers_all_exp[t]['expirations']
    interseс = set(pd.Series(dates).astype(str)) & set(pd.Series(expirations).astype(str))
    if len(interseс) == 0:
        continue

    final_df = pl.DataFrame()   
    with ThreadPoolExecutor(max_workers=24) as executor:
        futures = {
            executor.submit(get_option_eod, t, start, end): 
            (t, start, end) for start, end in zip(dates, dates)
        }
        for future in tqdm(as_completed(futures), total=len(dates)):
            try:
                cur_df = future.result()
                if (cur_df is None) or len(cur_df) == 0: 
                    continue
                final_df = pl.concat([final_df, cur_df])
            except Exception as e:
                print(cur_df)
                print(e)
                raise Exception
    try:
        if (final_df is None) or len(final_df) == 0: 
            continue
        final_df = final_df.sort('date', 'expiration', 'right', 'strike')\
            .with_columns(pl.col('date').cast(pl.String).str.strptime(pl.Date, '%Y%m%d', strict=False))
        for year in final_df['date'].dt.year().unique():
            cur_df = final_df.filter(pl.col('date').dt.year() == year)\
                .unique(subset=['expiration', 'date', 'right', 'strike'])
            min_date = cur_df['date'].min().strftime('%Y%m%d')
            max_date = cur_df['date'].max().strftime('%Y%m%d')

            cur_df.write_parquet(f'data/options/{min_date}_{max_date}_{t}_opts.parquet')
    except Exception as e:
        print(final_df)
        print(cur_df)
        print(e)
        raise Exception

  0%|          | 0/1681 [00:00<?, ?it/s]

  0%|          | 0/253 [00:00<?, ?it/s]

  0%|          | 0/241 [00:00<?, ?it/s]

 FRAN 20200103 - 20200103 don't have data 
 FRAN 20200102 - 20200102 don't have data 
 FRAN 20200106 - 20200106 don't have data 
 FRAN 20200107 - 20200107 don't have data 
 FRAN 20200108 - 20200108 don't have data 
 FRAN 20200113 - 20200113 don't have data 
 FRAN 20200114 - 20200114 don't have data 
 FRAN 20200109 - 20200109 don't have data 
 FRAN 20200110 - 20200110 don't have data 
 FRAN 20200115 - 20200115 don't have data 
 FRAN 20200121 - 20200121 don't have data 
 FRAN 20200122 - 20200122 don't have data 
 FRAN 20200116 - 20200116 don't have data 
 FRAN 20200123 - 20200123 don't have data 
 FRAN 20200117 - 20200117 don't have data 
 FRAN 20200127 - 20200127 don't have data 
 FRAN 20200124 - 20200124 don't have data 
 FRAN 20200130 - 20200130 don't have data 
 FRAN 20200128 - 20200128 don't have data 
 FRAN 20200203 - 20200203 don't have data 
 FRAN 20200129 - 20200129 don't have data 
 FRAN 20200205 - 20200205 don't have data 
 FRAN 20200131 - 20200131 don't have data 
 FRAN 20200

  0%|          | 0/253 [00:00<?, ?it/s]

...

  0%|          | 0/253 [00:00<?, ?it/s]

  0%|          | 0/253 [00:00<?, ?it/s]

  0%|          | 0/253 [00:00<?, ?it/s]

  0%|          | 0/253 [00:00<?, ?it/s]

  0%|          | 0/253 [00:00<?, ?it/s]

# Update EOD options

In [18]:
def get_option_eod(t: str, start_date: str, end_date: str):
    if end_date < '20160101':
        return None
        
    params = {
        'root': t, 'exp': 0,
        'start_date': start_date, 'end_date': end_date,  
        'use_csv': 'True'
    }
    ans, status_code = make_request('bulk_hist', 'option', 'eod', params)
    
    if status_code == 472:
        print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
        return None
    elif status_code == 572:
        print(f"\033[1;91m {t} an error \033[0m")
        return None
    
    ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))
    
    return ans_df

In [25]:
with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

need_year: str = '2024'
already_tickers = [file for file in os.listdir('data/options') if (need_year in file)][::-1]

already_tickers = [t for t in already_tickers if 'SPY_' in t] 

for ticker_file in tqdm(already_tickers):
    start_date = ticker_file.split('_')[0]
    end_date = ticker_file.split('_')[1]
    ticker = ticker_file.split('_')[2]

    # Find tickers without updates 
    if ticker in tickers_all_exp.keys():
        need_dates = pd.Series(
            tickers_all_exp[ticker]['trade_dates'],
            tickers_all_exp[ticker]['trade_dates']
        )[f'{need_year}0101':]
        
        earlest_dates = need_dates[:start_date][:-1]
        latest_dates = need_dates[end_date:][1:]
        if (len(earlest_dates) != 0) or (len(latest_dates) != 0):
            need_dates = pd.concat([earlest_dates, latest_dates])
        else:
            need_dates = None
        expirations = tickers_all_exp[ticker]['expirations']

    else:
        need_dates = pd.Series(
            tickers_all_exp['SPY']['trade_dates'],
            tickers_all_exp['SPY']['trade_dates']
        )[f'{need_year}0101':]
        
        earlest_dates = need_dates[:start_date][:-1]
        latest_dates = need_dates[end_date:][1:]
        if (len(earlest_dates) != 0) or (len(latest_dates) != 0):
            need_dates = pd.concat([earlest_dates, latest_dates])
        else:
            need_dates = None
        expirations = tickers_all_exp['SPY']['expirations']

    if need_dates is None:
        continue
        
    interseс = set(need_dates.values) & set(pd.Series(expirations).astype(str))
    if len(interseс) == 0:
        continue

    # Load new data
    final_df = pl.DataFrame()   
    with ThreadPoolExecutor(max_workers=24) as executor:
        futures = {
            executor.submit(get_option_eod, ticker, start, end): 
            (ticker, start, end) for start, end in zip(need_dates, need_dates)
        }
        for future in tqdm(as_completed(futures), total=len(need_dates)):
            try:
                cur_df = future.result()
                if (cur_df is None) or len(cur_df) == 0: 
                    continue
                final_df = pl.concat([final_df, cur_df])
            except Exception as e:
                print(cur_df)
                print(e)
                raise Exception
    try:
        if (final_df is None) or len(final_df) == 0: 
            continue
        final_df = final_df.sort('date', 'expiration', 'right', 'strike')\
            .with_columns(pl.col('date').cast(pl.String).str.strptime(pl.Date, '%Y%m%d', strict=False))\
            .unique(subset=['expiration', 'date', 'right', 'strike'])
        load_df = pl.read_parquet(f'data/options/{ticker_file}')
        final_df = pl.concat([final_df, load_df])

        for year in final_df['date'].dt.year().unique():
            cur_df = final_df.filter(pl.col('date').dt.year() == year)\
                .unique(subset=['expiration', 'date', 'right', 'strike'])
            min_date = cur_df['date'].min().strftime('%Y%m%d')
            max_date = cur_df['date'].max().strftime('%Y%m%d')

            cur_df.write_parquet(f'data/options/{min_date}_{max_date}_{ticker}_opts.parquet')
            
        os.remove(f'data/options/{ticker_file}')
        
    except Exception as e:
        print(final_df)
        print(load_df)
        print(e)
        raise Exception

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/43 [00:00<?, ?it/s]

# Update 1m stocks

In [3]:
def get_stock_1m(t: str, start_date: str, end_date: str):
    if end_date < '20160101':
        return None
        
    print(start_date, end_date)
    params = {
        'root': t, 'exp': 0,
        'start_date': start_date, 'end_date': end_date,  
        'use_csv': 'True', 'ivl': '60000'
    }
    ans, status_code = make_request('hist', 'stock', 'ohlc', params)
    
    if status_code == 472:
        print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
        return None
    elif status_code == 572:
        print(f"\033[1;91m {t} an error \033[0m")
        return None
    
    ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))
    
    return ans_df

In [7]:
start_range = '20160101'
end_range = datetime.datetime.now().strftime('%Y%m%d')

weekly_start = list(pd.date_range(start_range, end_range, freq='W-MON').strftime('%Y%m%d')[::-1])
weekly_end = list(pd.date_range(start_range, end_range, freq='W-SUN').strftime('%Y%m%d')[::-1])
weekly_end.extend([end_range])
weekly_end = np.sort(weekly_end)[::-1]

with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

already_tickers = [file for file in os.listdir('data/stocks/m1')][::-1]

    
need_tickers = ['SPY', 'GLD', 'IWM', 'QQQ', 'SLV', 'TLT', 'UNG', 'USO']
# need_tickers = ['SPY']
for t in tqdm(need_tickers, total=len(need_tickers)):
    final_df = pl.DataFrame()    
    start_time = time.time()

    already_df = None
    cur_file = [file for file in already_tickers if f'{t}_' in file]
    if len(cur_file) > 1: 
        print(cur_file)
        raise Exception
    elif len(cur_file) == 1: 
        cur_file = cur_file[0]
        already_df = pl.read_parquet(f'data/stocks/m1/{cur_file}')

        cur_w_start = [w for w in weekly_start if int(w) > already_df['date'].max()]
        cur_w_end = [w for w in weekly_end if int(w) > already_df['date'].max()]
        add_start = (pd.to_datetime(already_df['date'].max(), format='%Y%m%d') + pd.Timedelta(days=1)).strftime('%Y%m%d')
        add_end = (pd.to_datetime(cur_w_end[-1]) - pd.Timedelta(days=1)).strftime('%Y%m%d')
        cur_w_start.append(add_start)
        cur_w_end.append(add_end)
    else:
        cur_w_start = weekly_start
        cur_w_end = weekly_end

    # count = 0
    with ThreadPoolExecutor(max_workers=24) as executor:
        futures = {
            executor.submit(get_stock_1m, t, cur_s, cur_e): 
            (t, cur_s, cur_e) for cur_s, cur_e in zip(cur_w_start, cur_w_end)
        }
        for future in tqdm(as_completed(futures), total=len(cur_w_start)):
            cur_df = future.result()
            if cur_df is None: 
                continue
            final_df = pl.concat([final_df, cur_df])

            if len(final_df) == 0:
                count = 0
                continue

            # count += 1
            # if count > 10:
            #     final_df = final_df.sort('date')
            #     final_df.write_parquet(f'data/stocks/{start_range}_{end_range}_{t}_opts.parquet')
            #     count = 0

    if already_df is None:
        final_df = final_df.sort('date')
    else:
        final_df = pl.concat([already_df, final_df]).sort('date', 'ms_of_day')
    final_df.write_parquet(f'data/stocks/m1/{start_range}_{end_range}_{t}_opts.parquet')

  0%|          | 0/8 [00:00<?, ?it/s]

['20160101_20250326_SPY_opts.parquet', '20160101_20250127_SPY_opts.parquet']


Exception: 

# Update 1m Options

In [3]:
ticker: str = 'SPY'

In [4]:
with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

expirs = pd.Series(data=tickers_all_exp[ticker]['expirations'], index=tickers_all_exp[ticker]['expirations'])
min_date = tickers_all_exp[ticker]['trade_dates'][0]
max_date = tickers_all_exp[ticker]['trade_dates'][-1]
expirs = expirs.loc[min_date:max_date]

expirs = pd.DataFrame(expirs, columns=['expir'])
expirs['expir'] = expirs['expir'].astype(str)
expirs['ticker'] = ticker

expirs['dayOfWeek'] = pd.to_datetime(expirs['expir']).dt.dayofweek
expirs = expirs[~expirs['dayOfWeek'].isin([5, 6])]

expirs.groupby('dayOfWeek').agg(cnt=('ticker', 'count')).sort_index()

,cnt
dayOfWeek,
0,235
1,131
2,261
3,121
4,256
